In [47]:
import pennylane as qml
from pennylane import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [48]:
iris = load_iris()
X = iris.data
Y = iris.target

In [49]:
print(X)
print(Y)

[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]
 [5.4 3.7 1.5 0.2]
 [4.8 3.4 1.6 0.2]
 [4.8 3.  1.4 0.1]
 [4.3 3.  1.1 0.1]
 [5.8 4.  1.2 0.2]
 [5.7 4.4 1.5 0.4]
 [5.4 3.9 1.3 0.4]
 [5.1 3.5 1.4 0.3]
 [5.7 3.8 1.7 0.3]
 [5.1 3.8 1.5 0.3]
 [5.4 3.4 1.7 0.2]
 [5.1 3.7 1.5 0.4]
 [4.6 3.6 1.  0.2]
 [5.1 3.3 1.7 0.5]
 [4.8 3.4 1.9 0.2]
 [5.  3.  1.6 0.2]
 [5.  3.4 1.6 0.4]
 [5.2 3.5 1.5 0.2]
 [5.2 3.4 1.4 0.2]
 [4.7 3.2 1.6 0.2]
 [4.8 3.1 1.6 0.2]
 [5.4 3.4 1.5 0.4]
 [5.2 4.1 1.5 0.1]
 [5.5 4.2 1.4 0.2]
 [4.9 3.1 1.5 0.2]
 [5.  3.2 1.2 0.2]
 [5.5 3.5 1.3 0.2]
 [4.9 3.6 1.4 0.1]
 [4.4 3.  1.3 0.2]
 [5.1 3.4 1.5 0.2]
 [5.  3.5 1.3 0.3]
 [4.5 2.3 1.3 0.3]
 [4.4 3.2 1.3 0.2]
 [5.  3.5 1.6 0.6]
 [5.1 3.8 1.9 0.4]
 [4.8 3.  1.4 0.3]
 [5.1 3.8 1.6 0.2]
 [4.6 3.2 1.4 0.2]
 [5.3 3.7 1.5 0.2]
 [5.  3.3 1.4 0.2]
 [7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.

In [50]:
x_train,x_test,y_train,y_test = train_test_split(X,Y)

In [51]:
x_mean = np.mean(x_train,axis=0)
x_std = np.std(x_train,axis=0)
x_train = (x_train-x_mean)/x_std
x_test = (x_test-x_mean)/x_std

In [52]:
def mapping(x):
    if x==0:
        return [1.0,1.0]
    elif x==1:
        return [-1.0,1.0]
    return [-1.0,-1.0]
y_mapped = np.array([mapping(y) for y in y_train],requires_grad = False)

In [53]:
devices = qml.device('default.qubit',wires=2)
def quantum_circuit(x,parameter):
    qml.RX(x[0],wires=0)
    qml.RY(x[1],wires=0)
    qml.RX(x[2],wires=1)
    qml.RY(x[3],wires=1)

    qml.RX(parameter[0],wires=0)
    qml.RY(parameter[1],wires=0)
    qml.RX(parameter[2],wires=1)
    qml.RY(parameter[3],wires=1)
    qml.CNOT(wires=[0,1])
    qml.RX(parameter[4],wires=0)
    qml.RY(parameter[5],wires=1)

    qml.RX(parameter[6], wires=0)
    qml.RY(parameter[7], wires=0)
    qml.RX(parameter[8], wires=1)
    qml.RY(parameter[9], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RX(parameter[10], wires=0)
    qml.RY(parameter[11], wires=1)
    return [qml.expval(qml.PauliZ(0)),qml.expval(qml.PauliZ(1))]

In [54]:
circuit = qml.QNode(quantum_circuit,devices)

In [55]:
def cost(params,x_data,y_data):
    pred = np.array([circuit(x,params) for x in x_data])
    return np.mean((pred-y_data)**2)

In [56]:
def predict_class(pred):
    z = {0:[1.0,1.0] , 1:[-1.0,1.0] , 2:[-1.0,-1.0]}
    ans = -1
    distance=10000000000
    for label,target in z.items():
        dis = np.linalg.norm(pred-target)
        if dis<distance:
            distance = dis
            ans = label
    return ans
def accuracy(params,x_data,y_data):
    pred = np.array([circuit(x,params) for x in x_data])
    pred = np.array([predict_class(prediction) for prediction in pred])
    ans = np.sum(pred==y_data)
    ans=ans/(len(y_data))
    return ans

In [58]:
np.random.seed(40)
params = np.random.randn(12,requires_grad = True)
optimiser = qml.GradientDescentOptimizer(stepsize=0.1)
epochs = 100
batch_size = 20
for epoch in range(epochs):
    batch_index = np.random.randint(0, len(x_train), (batch_size,))
    X_batch = x_train[batch_index]
    Y_batch = y_mapped[batch_index]
    params = optimiser.step(lambda v: cost(v, X_batch, Y_batch), params)
    if (epoch+1) % 10 == 0:
        current_cost = cost(params,X_batch,Y_batch)
        accu = accuracy(params,X_batch,y_train[batch_index])
        print(f'cost is {current_cost} and current accuracy is {accu*100}')
print(f'final paramters are {params}')

cost is 0.8941348251127739 and current accuracy is 50.0
cost is 0.6116982750231632 and current accuracy is 70.0
cost is 0.3568638096140594 and current accuracy is 75.0
cost is 0.5578788668217325 and current accuracy is 70.0
cost is 0.30685891055160824 and current accuracy is 90.0
cost is 0.2134104790825278 and current accuracy is 90.0
cost is 0.3273200545557067 and current accuracy is 95.0
cost is 0.22206557839377875 and current accuracy is 95.0
cost is 0.25539343859586505 and current accuracy is 95.0
cost is 0.17594813531017478 and current accuracy is 100.0
final paramters are [ 0.80579572 -1.03675432 -0.57337374  1.11213039 -2.03482776 -0.15885618
  2.10206362 -0.16178124  1.07233492  0.24886358  0.81575375 -0.00757969]


In [59]:
test_pred = np.array([predict_class(np.array(circuit(x,params))) for x in x_test])
print(x_test)
print(test_pred)

[[ 0.84377643 -0.13310314  1.03624729  0.81806763]
 [ 0.60146115 -1.73853172  0.42247005  0.17734333]
 [ 0.72261879 -0.59179702  1.09204522  1.20250221]
 [ 0.35914586 -0.13310314  0.53406591  0.30548819]
 [-1.70053403 -0.13310314 -1.30726581 -1.23225012]
 [-0.97358818  0.78428463 -1.13987202 -0.9759604 ]
 [ 0.60146115 -0.36245008  1.09204522  0.81806763]
 [-0.36779998 -1.73853172  0.19927832  0.17734333]
 [ 1.32840699  0.09624381  0.7014597   0.43363305]
 [ 0.60146115 -0.59179702  0.81305557  0.43363305]
 [ 2.29766812  1.70167239  1.70582246  1.33064707]
 [ 0.72261879 -0.36245008  0.36667212  0.17734333]
 [ 0.60146115 -1.27983784  0.7014597   0.43363305]
 [-0.00432706 -0.82114396  0.14348039  0.04919847]
 [-0.36779998  2.61906016 -1.25146788 -1.23225012]
 [ 0.60146115  0.55493769  1.31523694  1.71508165]
 [-0.24664234 -0.13310314  0.47826798  0.43363305]
 [ 0.72261879 -0.82114396  0.92465143  0.94621249]
 [ 0.23798822  0.78428463  0.47826798  0.56177791]
 [-0.97358818  1.24297851 -1.25

In [60]:
accu = np.sum(test_pred==y_test)/len(y_test)
print(f'accuracy on test set is {accu*100}%')

accuracy on test set is 89.47368421052632%
